### Introduction
This notebook demonstrates how to use SRA public datasets hosted by AWS to get RNA-Seq gene counts file locations and their analysis objects attributes/QC metrics using Athena.   
It uses boto3 and awswrangler python AWS clients instead of CLI tools.  

### AWS account and credentials
To run this tutorial you need:
- AWS account  
- role to use with appropriate permissions set, which is out of scope of this tutorial  
- and a file containning said role's Access Key ID and Secret Access Key in the following format:
```
$cat path/to/your/aws-credentials.txt
akid=<access-key-id>
sak=<secret-access-key>
```

### Source data location
**sra-pub-metadata-us-east-1** is AWS S3 public data bucket in **us-east-1** region containing gziped parquet files for  
- metadata table in sra/metadata subfolder
- analysis table in sra/analysis subfolder 

In [94]:
s3_bucket='sra-pub-metadata-us-east-1'
# aws regions is nailed down by source data location
aws_region='us-east-1'

### Setup

In [1]:
%%capture
%env PIP_DISABLE_PIP_VERSION_CHECK=1

In [2]:
!pip3 install -q boto3[crt] awswrangler

In [44]:
import boto3
import botocore
import pandas as pd
import awswrangler as wr
import awswrangler.athena as wa
import time
import os
from collections import namedtuple
from datetime import datetime
pd.set_option('display.max_colwidth', None)

#### Loading AWS credentials

In [10]:
def get_aws_credentials(file: str = 'aws-credentials.txt'):
    with open(file, "rt") as _in:
        return dict(line.strip().split("=", 1) for line in _in if "=" in line and not line.startswith('#'))

In [95]:
aws_credentials=get_aws_credentials()
aws_credentials=namedtuple("aws_credentials", aws_credentials.keys())(**aws_credentials)

### Specify Athena database name and output bucket
In case the role you are using does not have enough permissions to create new database please use existing one

In [55]:
aws_database='sra_athena_tutorial'
# output folder is required for Athena queries
aws_output = 's3://sra-staging-metadata-us-east-1/sra-athena-tutorial-output/'

### Creating Athena client

In [18]:
aws_session = boto3.Session(aws_access_key_id=aws_credentials.akid, aws_secret_access_key=aws_credentials.sak, region_name=aws_region)

s3 = boto3.client('s3', aws_access_key_id=aws_credentials.akid, aws_secret_access_key=aws_credentials.sak, region_name=aws_region)
ac = boto3.client('athena', aws_access_key_id=aws_credentials.akid, aws_secret_access_key=aws_credentials.sak, region_name=aws_region)

### Optional, if permissions allow for it, creating database

In [56]:
try:
    wr.catalog.create_database(name=aws_database, boto3_session=aws_session, exist_ok=True, description="SRA Athena Tutorial")
except botocore.exceptions.ClientError as ex:
    error_code = ex.response.get("Error", {}).get("Code")
    if error_code in ["AccessDenied", "AccessDeniedException"]:    
        print('You do not have permissions to create new database')
    else:
        print('oops')

### Creating *analysis* table if not exist

In [57]:
wr.catalog.create_parquet_table(
    database=aws_database,
    table="sra_analysis",
    path="s3://sra-pub-metadata-us-east-1/sra/analysis/",
    columns_types={
    	"acc": "STRING",
    	"study": "STRING",
    	"biosample": "STRING",
    	"bioproject": "STRING",
    	"alias": "STRING",
    	"center_name": "STRING",
    	"analysis_type": "STRING",
    	"target_run": "STRING",
    	"alignment": "STRUCT<includes_unaligned_reads:BOOLEAN, marks_duplicate_reads:BOOLEAN, includes_failed_reads:BOOLEAN>",
    	"standard_assembly": "STRING",
    	"pipe_section": "STRING",
    	"attributes": "ARRAY<STRUCT<k:STRING,v:STRING>>",
    	"files_processed": "BOOLEAN"
    },
    mode="overwrite",
    compression="gzip",
    boto3_session=aws_session    
)

In [96]:
sql = "select count(*) c from sra_analysis"
df = wa.read_sql_query(sql=sql, database=aws_database, boto3_session=aws_session, s3_output = aws_output)
print(f"{df.c[0]:,}")

4,576,347


### Creating *metadata* table if not exist

In [60]:
wr.catalog.create_parquet_table(
    database=aws_database,
    table="sra_metadata",
    path="s3://sra-pub-metadata-us-east-1/sra/metadata/",
    columns_types={
    	"acc": "STRING",
    	"assay_type": "STRING",
    	"center_name": "STRING",
    	"consent": "STRING",
    	"experiment": "STRING",
    	"sample_name": "STRING",
    	"instrument": "STRING",
    	"librarylayout": "STRING",
    	"libraryselection": "STRING",
    	"librarysource": "STRING",
    	"platform": "STRING",
    	"sample_acc": "STRING",
    	"biosample": "STRING",
    	"organism": "STRING",
    	"sra_study": "STRING",
    	"releasedate": "DATE",
    	"bioproject": "STRING",
    	"mbytes": "BIGINT",
    	"loaddate": "TIMESTAMP",
    	"avgspotlen": "BIGINT",
    	"mbases": "BIGINT",
    	"insertsize": "BIGINT",
    	"library_name": "STRING",
    	"biosamplemodel_sam": "ARRAY<STRING>",
    	"collection_date_sam": "DATE",
    	"geo_loc_name_country_calc": "STRING",
    	"geo_loc_name_country_continent_calc": "STRING",
    	"geo_loc_name_sam": "ARRAY<STRING>",
    	"ena_first_public_run": "ARRAY<STRING>",
    	"ena_last_update_run": "ARRAY<STRING>",
    	"sample_name_sam": "ARRAY<STRING>",
    	"datastore_filetype": "ARRAY<STRING>",
    	"datastore_provider": "ARRAY<STRING>",
    	"datastore_region": "ARRAY<STRING>",
    	"attributes": "ARRAY<STRUCT<k:STRING,v:STRING>>",
    	"run_file_version": "BIGINT",
    	"jattr": "STRING"
    },
    mode="overwrite",
    compression="gzip",
    boto3_session=aws_session    
)

In [97]:
sql = "select count(*) c from sra_metadata"
df = wa.read_sql_query(sql=sql, database=aws_database, boto3_session=aws_session, s3_output = aws_output)
print(f"{df.c[0]:,}")

43,720,379


### Getting list of gene counts accessions and their locations by BioProject and optionally other metadata attributes
Though BioProject exists in analysis it is chosen here for illustration

In [66]:
bioproject='PRJNA432002'

##### Either with pivoted attributes in order to further narrow down by say alignment rate

In [91]:
sql=f"""
with X as (
SELECT 
acc, target_run, map_agg(item.k, case when item.k='alignment_rate' then split(item.v, '%')[1] else item.v end) as kv
FROM 
    sra_analysis cross join unnest(attributes) a(item) 
GROUP BY acc, target_run
), A as (
select 
    acc, target_run,
    kv['total_reads'] total_reads, 
    kv['layout'] layout,
    kv['intronic_reads'] intronic_reads,
    kv['intergenic_reads'] intergenic_reads,
    kv['mito_count'] mito_count,
    kv['decay_53_ratio'] decay_53_ratio,
    kv['bases'] bases,
    kv['bio_bases'] bio_bases,
    kv['strandness'] strandness,
    kv['alignment_rate'] alignment_rate,
    kv['unaligned_reads'] unaligned_reads,
    kv['discordant_pairs'] discordant_pairs
from X
),
Z as (  
  SELECT m.acc as run, 's3://sra-rnaseq-analysis/tsv/'||A.acc||'/'||m.acc||'.ncbi-gene-counts.tsv' path, a.*
  FROM sra_metadata m left join A on A.target_run = m.acc
  where m.bioproject = '{bioproject}'
)
select * from Z
;
"""
dfa = wa.read_sql_query(sql=sql, database=aws_database, boto3_session=aws_session, s3_output = aws_output)

##### Available analysis QC metrics and attributes

In [92]:
dfa.head(1).T

,0
run,SRR6570861
path,s3://sra-rnaseq-analysis/tsv/SRZ2896535/SRR6570861.ncbi-gene-counts.tsv
acc,SRZ2896535
target_run,SRR6570861
total_reads,654432
layout,PAIRED
intronic_reads,73238
intergenic_reads,40916
mito_count,15341
decay_53_ratio,0.0738


#### Note - it is possible not all runs will have gene counts available

In [93]:
print(dfa.path.isnull().sum())

8877


#### Or downloading attributes as json if needed

In [109]:
sql = f"""
with A as (
    SELECT 
        acc, target_run, map_agg(item.k, case when item.k='alignment_rate' then split(item.v, '%')[1] else item.v end) as kv
    FROM 
        sra_analysis cross join unnest(attributes) a(item) 
    GROUP BY acc, target_run
)
SELECT m.acc as run, 's3://sra-rnaseq-analysis/tsv/'||A.acc||'/'||m.acc||'.ncbi-gene-counts.tsv' path, A.acc, json_format(cast(kv as JSON)) json
FROM sra_metadata m left join A on A.target_run = m.acc
where m.bioproject = '{bioproject}'
;
"""
dfb = wa.read_sql_query(sql=sql, database=aws_database, boto3_session=aws_session, s3_output = aws_output)

In [110]:
df=dfb.head(1).T
df.style.set_properties(**{"text-align": "left"})

,0
run,SRR6611909
path,s3://sra-rnaseq-analysis/tsv/SRZ3268819/SRR6611909.ncbi-gene-counts.tsv
acc,SRZ3268819
json,"{""alignment_rate"":""88.09"",""bases"":""200557800"",""bio_bases"":""200557800"",""decay_53_ratio"":""0.1653"",""discordant_pairs"":""9465"",""intergenic_reads"":""43332"",""intronic_reads"":""149257"",""layout"":""PAIRED"",""mito_count"":""18660"",""strandness"":""none"",""total_reads"":""1002789"",""unaligned_reads"":""187441""}"
